# Ensembl REST

Two tools share the `ensembl` toolkit:

- **`ensembl-fetch`** — gene / transcript / exon hierarchy lookup, sequence retrieval, region overlap, and cross-references. One tool with an `endpoint:` switch over five fetch endpoints.
- **`ensembl-vep`** — variant-effect prediction from HGVS notation.

Both call `rest.ensembl.org` (GRCh38) by default; `assembly="GRCh37"` routes to `grch37.rest.ensembl.org`.

In [ ]:
from proto_tools.tools.database_retrieval import (
    EnsemblFetchConfig,
    EnsemblFetchInput,
    EnsemblVEPConfig,
    EnsemblVEPInput,
    run_ensembl_fetch,
    run_ensembl_vep,
)

## ensembl-fetch — Input

| Field | Type | Description |
|---|---|---|
| `ensembl_id` | `str \| None` | Ensembl ID (`ENSG...`, `ENST...`, `ENSP...`). |
| `symbol` | `str \| None` | Gene symbol (e.g. `BRCA1`). |

Provide whichever the chosen endpoint requires.

## ensembl-fetch — Config

| Field | Type | Default | Description |
|---|---|---|---|
| `endpoint` | `Literal["lookup_id","lookup_symbol","sequence","overlap","xrefs"]` | `"lookup_id"` | Which Ensembl REST endpoint. |
| `species` | `Literal[5]` | `"homo_sapiens"` | Used by `lookup_symbol` + symbol-form `xrefs`. |
| `assembly` | `Literal["GRCh38","GRCh37"]` | `"GRCh38"` | GRCh37 routes to `grch37.rest.ensembl.org`. |
| `sequence_type` | `Literal["genomic","cdna","cds","protein"]` | `"cdna"` | `sequence` endpoint only. |
| `overlap_feature` | `Literal[...]` | `"gene"` | `overlap` endpoint only. |
| `expand` | `bool` | `True` | Expand transcripts/exons (`lookup_*` only). |

## Lookup BRCA1 by symbol

In [ ]:
out = run_ensembl_fetch(
    EnsemblFetchInput(symbol="BRCA1"),
    EnsemblFetchConfig(endpoint="lookup_symbol"),
)
assert out.success
gene = out.result
print(f"{gene.display_name} ({gene.id}): {len(gene.Transcript)} transcripts; canonical={gene.canonical_transcript}")

## Fetch the protein sequence of the canonical transcript

In [ ]:
canonical_id = gene.canonical_transcript.split(".")[0]
seq_out = run_ensembl_fetch(
    EnsemblFetchInput(ensembl_id=canonical_id),
    EnsemblFetchConfig(endpoint="sequence", sequence_type="protein"),
)
print(f"{seq_out.result.id}: {len(seq_out.result.seq)} aa")

## Cross-reference Ensembl → UniProt

In [ ]:
xrefs_out = run_ensembl_fetch(
    EnsemblFetchInput(ensembl_id=gene.id),
    EnsemblFetchConfig(endpoint="xrefs"),
)
uniprot_id = next(x.primary_id for x in xrefs_out.result if x.dbname == "Uniprot_gn")
print(f"UniProt accession for {gene.display_name}: {uniprot_id}")

## Variant Effect Prediction

HGVS forms supported: genomic (`9:g.22125504G>C`), coding (`ENST00000357654:c.5074G>A`), or protein (`ENSP00000418960:p.Tyr124Cys`).

In [ ]:
vep = run_ensembl_vep(EnsemblVEPInput(hgvs="9:g.22125504G>C"))
top = vep.consequences[0]
print(f"{top.allele_string}: {top.most_severe_consequence}, {len(top.transcript_consequences)} transcripts affected")

## GRCh37 — same gene, different coordinates

Set `assembly="GRCh37"` for legacy/clinical datasets pinned to the older assembly.

In [ ]:
grch37 = run_ensembl_fetch(
    EnsemblFetchInput(ensembl_id=gene.id),
    EnsemblFetchConfig(endpoint="lookup_id", assembly="GRCh37"),
)
print(f"GRCh38 start: {gene.start} -> GRCh37 start: {grch37.result.start}")